<span style = "font-family: Verdana; font-size: 20px">

#### **Feature Engineering**
</span>

In [1]:
import numpy as np
import pandas as pd

In [2]:
data = pd.read_csv("data/modified/viz_pokemon.csv", keep_default_na=False, dtype = {'ID': str})

In [3]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 54 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   ID                  800 non-null    object 
 1   Name                800 non-null    object 
 2   Type 1              800 non-null    object 
 3   Type 2              800 non-null    object 
 4   HiddenAbility       800 non-null    object 
 5   Generation          800 non-null    int64  
 6   Hp                  800 non-null    float64
 7   Attack              800 non-null    float64
 8   Defense             800 non-null    float64
 9   SpecialAttack       800 non-null    float64
 10  SpecialDefense      800 non-null    float64
 11  Speed               800 non-null    float64
 12  TotalStats          800 non-null    float64
 13  Weight              800 non-null    float64
 14  Height              800 non-null    float64
 15  GenderProbM         800 non-null    float64
 16  Category

<span style = "font-family: Verdana; font-size: 20px">

Dummy hoá các cột categorical (`Type` và `EggGroup`) thành số để sử dụng trong mô hình ML.
</span>

In [4]:
type1dum = pd.get_dummies(data['Type 1'], prefix='Type', dtype=int)
type2dum = pd.get_dummies(data['Type 2'], prefix='Type', dtype=int)

dummies_total = type1dum.add(type2dum, fill_value=0)

print(dummies_total)

data = pd.concat([data, dummies_total], axis=1)
data = data.drop(['Type 1', 'Type 2'], axis=1)


egggroup1dum = pd.get_dummies(data['EggGroup1'], prefix='Egg', dtype=int)
egggroup2dum = pd.get_dummies(data['EggGroup2'], prefix='Egg', dtype=int)

dummies_total = egggroup1dum.add(egggroup2dum, fill_value=0)

print(dummies_total)

data = pd.concat([data, dummies_total], axis=1)
data = data.drop(['EggGroup1', 'EggGroup2'], axis=1)

     Type_Bug  Type_Dark  Type_Dragon  Type_Electric  Type_Fairy  \
0           0          0            0              0           0   
1           0          0            0              0           0   
2           0          0            0              0           0   
3           0          0            0              0           0   
4           0          0            0              0           0   
..        ...        ...          ...            ...         ...   
795         0          0            0              0           1   
796         0          0            0              0           1   
797         0          0            0              0           0   
798         0          1            0              0           0   
799         0          0            0              0           0   

     Type_Fighting  Type_Fire  Type_Flying  Type_Ghost  Type_Grass  \
0                0          0            0           0           1   
1                0          0            0 

<span style = "font-family: Verdana; font-size: 20px">

Chuyển `LevelingRate` thành các giá trị số tương ứng để sử dụng trong mô hình ML.

[Nguồn tham khảo](https://bulbapedia.bulbagarden.net/wiki/Experience)
</span>

In [5]:
data['LevelingRate'] = data['LevelingRate'].replace(['Slow', 'Medium Slow', 'Medium Fast', 'Fast', 'Erratic', 'Fluctuating'], [1250000, 1059860, 1000000, 800000, 600000, 1640000]).infer_objects(copy=False)

C:\Users\Administrator\AppData\Local\Temp\ipykernel_11304\4046927376.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data['LevelingRate'] = data['LevelingRate'].replace(['Slow', 'Medium Slow', 'Medium Fast', 'Fast', 'Erratic', 'Fluctuating'], [1250000, 1059860, 1000000, 800000, 600000, 1640000]).infer_objects(copy=False)


<span style = "font-family: Verdana; font-size: 20px">

#### **Feature Creation**
</span>

In [ ]:
def engineer_top_10_features(data):
    """
    Creates the 10 most critical features for predicting Pokemon victory.
    Focuses on Game Mechanics, Stat Efficiency, and Type Matchups.
    """
    # Tạo bản sao để tránh cảnh báo SettingWithCopy
    data = data.copy()

    # --- QUAN TRỌNG: Phải tính Alpha_Strike ĐẦU TIÊN ---
    # (Để các công thức bên dưới có dữ liệu mà dùng)
    data['Alpha_Strike'] = data['Attack'] + data['SpecialAttack']

    # --- FEATURE 2: Durability Score (Khả Năng Chịu Đựng) ---
    data['Durability_Score'] = (data['Hp'] * (data['Defense'] + data['SpecialDefense'])) / 100

    # --- FEATURE 4: Glass Cannon Index (Chỉ Số "Máu Giấy") ---
    # Bây giờ cột Alpha_Strike đã tồn tại, dòng này sẽ không bị lỗi nữa
    data['Glass_Cannon_Index'] = data['Alpha_Strike'] / (data['Defense'] + data['SpecialDefense'] + 1)

    # --- FEATURE 5: Type Vulnerability (Tổng Mức Độ Bị Khắc Chế) ---
    dmg_cols = [c for c in data.columns if c.startswith('DamageFrom')]
    data['Type_Vulnerability'] = data[dmg_cols].apply(lambda row: sum(x for x in row if x > 1.0), axis=1)

    # --- FEATURE 6: Insta-Kill Risk (Nguy Cơ Tử Huyệt) ---
    data['Insta_Kill_Risk'] = (data[dmg_cols] >= 4.0).sum(axis=1)

    # --- FEATURE 8: Tank Bias (Thiên Hướng Phòng Thủ) ---
    data['Tank_Bias'] = (data['Defense'] + data['SpecialDefense']) / data['TotalStats']

    # --- FEATURE 9: Versatility Bonus (Lợi Thế Đa Hệ) ---
    # Logic: Tổng các cột dummy Type_. Nếu > 1 nghĩa là Pokemon đa hệ.
    type_cols = [c for c in data.columns if c.startswith('Type_')]
    data['Versatility_Bonus'] = data[type_cols].sum(axis=1).apply(lambda x: 1 if x > 1 else 0)

    # --- FEATURE 10: Legendary Factor (Hệ Số Huyền Thoại) ---
    data['Legendary_Factor'] = data['IsLegendary'] + data['IsMythical'] + data['HasMega']

    return data

# --- EXECUTION ---

if 'data' in locals():
    print("Bắt đầu Feature Engineering...")
    original_features = data.columns.tolist()
    
    # Chạy hàm tạo feature
    data = engineer_top_10_features(data)

    # --- REPORTING ---
    new_features = [col for col in data.columns if col not in original_features]
    print("--- Feature Engineering Report ---")
    print(f"Đã tạo thành công {len(new_features)} đặc trưng mới.")
    print("Danh sách các feature mới:")
    for feature in new_features:
        print(f"- {feature}")
    print("---------------------------------\n")

    print(data.head())

    # --- SAVE TO SINGLE FILE ---
    output_path = 'viz_pokemon_engineered.csv'
    data.to_csv(output_path, index=False)
    print(f"Đã lưu toàn bộ dữ liệu vào file duy nhất: {output_path}")

else:
    print("Lỗi: Biến 'data' chưa tồn tại. Vui lòng chạy các cell phía trên trước.")

Bắt đầu Feature Engineering...
--- Feature Engineering Report ---
Đã tạo thành công 0 đặc trưng mới.
Danh sách các feature mới:
---------------------------------

  ID           Name HiddenAbility  Generation    Hp  Attack  Defense  \
0  1      Bulbasaur   Chlorophyll           1  45.0    49.0     49.0   
1  2        Ivysaur   Chlorophyll           1  60.0    62.0     63.0   
2  3       Venusaur   Chlorophyll           1  80.0    82.0     83.0   
3  4  Mega Venusaur          None           1  80.0   100.0    123.0   
4  5     Charmander   Solar Power           1  39.0    52.0     43.0   

   SpecialAttack  SpecialDefense  Speed  ...  Egg_Water 2', 'Dragon  \
0           65.0            65.0   45.0  ...                    0.0   
1           80.0            80.0   60.0  ...                    0.0   
2          100.0           100.0   80.0  ...                    0.0   
3          122.0           120.0   80.0  ...                    0.0   
4           60.0            50.0   65.0  ...     

In [6]:
data.columns

Index(['ID', 'Name', 'HiddenAbility', 'Generation', 'Hp', 'Attack', 'Defense',
       'SpecialAttack', 'SpecialDefense', 'Speed', 'TotalStats', 'Weight',
       'Height', 'GenderProbM', 'Category', 'CatchRate', 'EggCycles',
       'LevelingRate', 'BaseFriendship', 'IsLegendary', 'IsMythical',
       'HasMega', 'EvoStage', 'TotalEvoStages', 'PreevoName',
       'DamageFromNormal', 'DamageFromFighting', 'DamageFromFlying',
       'DamageFromPoison', 'DamageFromGround', 'DamageFromRock',
       'DamageFromBug', 'DamageFromGhost', 'DamageFromSteel', 'DamageFromFire',
       'DamageFromWater', 'DamageFromGrass', 'DamageFromElectric',
       'DamageFromPsychic', 'DamageFromIce', 'DamageFromDragon',
       'DamageFromDark', 'DamageFromFairy', 'Ability 1', 'Ability 2',
       'NoGender', 'IsMega', 'FightsWon', 'TotalFights', 'WinPercentage',
       'Type_Bug', 'Type_Dark', 'Type_Dragon', 'Type_Electric', 'Type_Fairy',
       'Type_Fighting', 'Type_Fire', 'Type_Flying', 'Type_Ghost', 'Type_Gras

In [7]:
with open('data/modified/pokemonML.csv', 'w', encoding = 'utf-8') as f:
    data.to_csv(f, index=False)